# Line Feature Demo

This notebook demonstrates the **line feature matching pipeline** provided by the
LineExtraction library.  It covers:

1. Detecting line segments with LSD (CC detector)
2. Computing **LBD** descriptors using the LSD detector's native Sobel gradients
   (72-dim float, L2 distance)
3. Computing **OpenCV LBD** descriptors (256-bit binary, Hamming distance) via the
   `FdcOpenCVLBD` C++ wrapper
4. **Brute-force matching** with cross-check and ratio test filtering
5. **Ground-truth evaluation** — matches are color-coded green (correct) or red (incorrect)
6. **Runtime comparison** between both descriptor types
7. Comparing both descriptor types interactively with **Rerun**

## Prerequisites

```bash
# Build all Python bindings (includes FdcOpenCVLBD and BruteForceOpenCVLBD)
bazel build //libs/...

# Install notebook deps (in the project .venv)
pip install rerun-sdk[notebook] Pillow

# Download HPatches evaluation dataset (recommended)
./tools/scripts/setup_hpatches.sh
```

> By default the notebook uses the **HPatches** `v_london` sequence — a
> London cityscape with real perspective change and abundant line structure.
> Change `HPATCHES_SEQ`, `HPATCHES_TARGET`, and `N_EVAL` in the config cell
> to experiment with different sequences and difficulty levels.
> Set `USE_HPATCHES = False` to fall back to a synthetic perspective warp
> of the windmill image.

## 1. Setup & Imports

In [ ]:
import sys
import pathlib
import colorsys

import numpy as np
import rerun as rr
from PIL import Image

# --- Discover workspace root and add binding paths ---
def _find_workspace_root() -> pathlib.Path:
    """Walk up from CWD until we find MODULE.bazel."""
    p = pathlib.Path.cwd()
    while p != p.parent:
        if (p / "MODULE.bazel").exists():
            return p
        p = p.parent
    raise RuntimeError("Cannot find workspace root (MODULE.bazel)")

ROOT = _find_workspace_root()
for lib in ["geometry", "imgproc", "edge", "lsd", "lfd", "algorithm", "eval"]:
    sp = str(ROOT / "bazel-bin" / "libs" / lib / "python")
    if sp not in sys.path:
        sys.path.insert(0, sp)
# Also add the top-level python/ package
sp = str(ROOT / "python")
if sp not in sys.path:
    sys.path.insert(0, sp)

import le_lsd
import le_lfd
from lsfm.data import TestImages, HPATCHES_LINE_SEQUENCES

print(f"Workspace: {ROOT}")
print(f"le_lsd types:  {[x for x in dir(le_lsd) if x.startswith('Lsd')]}")
print(f"le_lfd types:  {[x for x in dir(le_lfd) if not x.startswith('_')]}")

## 2. Load Image Pair

We try HPatches first.  If it's not available, we synthetically warp the
windmill test image with a random perspective homography.

In [ ]:
import cv2
import time

images = TestImages()

def _synth_homography(img: np.ndarray, seed: int = 42,
                      perturbation: float = 0.10) -> tuple[np.ndarray, np.ndarray]:
    """Generate a synthetic perspective warp + ground-truth H.

    Parameters
    ----------
    img : np.ndarray
        Grayscale input image.
    seed : int
        RNG seed for reproducibility.
    perturbation : float
        Corner perturbation as fraction of image size (default 10%).
    """
    rng = np.random.default_rng(seed)
    h, w = img.shape[:2]
    src = np.float32([[0, 0], [w, 0], [w, h], [0, h]])
    offsets = (rng.uniform(-perturbation, perturbation, src.shape).astype(np.float32)
               * np.array([w, h], dtype=np.float32))
    dst = src + offsets
    H = cv2.getPerspectiveTransform(src, dst)
    warped = cv2.warpPerspective(img, H, (w, h), borderMode=cv2.BORDER_REFLECT)
    return warped, H

def _downscale(img: np.ndarray, max_dim: int = 800) -> tuple[np.ndarray, float]:
    """Downscale image so the longest side <= max_dim. Returns (image, scale)."""
    h, w = img.shape[:2]
    if max(h, w) <= max_dim:
        return img, 1.0
    scale = max_dim / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    return cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA), scale

# ================================================================
# Configuration — change these to experiment with different images
# ================================================================
USE_HPATCHES = True             # True to use HPatches pairs (recommended)
HPATCHES_SEQ = "v_london"       # HPatches sequence name (v_ = viewpoint)
HPATCHES_TARGET = 3             # target image index (2-6), higher = harder
MAX_DIM = 800                   # downscale so longest side <= this
MIN_SEG_LENGTH = 15             # minimum segment length in pixels
RATIO_THRESH = 0.85             # Lowe's ratio test threshold
N_EVAL = 100                    # how many top matches to show & evaluate
# ================================================================

if USE_HPATCHES:
    try:
        ref_path, tgt_path, H_gt = images.hpatches_pair(
            HPATCHES_SEQ, target_idx=HPATCHES_TARGET)
        img_ref = np.array(Image.open(ref_path).convert("L"))
        img_tgt = np.array(Image.open(tgt_path).convert("L"))
        # Downscale for speed + memory safety; adjust homography accordingly
        img_ref, s_ref = _downscale(img_ref, MAX_DIM)
        img_tgt, s_tgt = _downscale(img_tgt, MAX_DIM)
        S_ref = np.diag([s_ref, s_ref, 1.0])
        S_tgt = np.diag([s_tgt, s_tgt, 1.0])
        H_gt = S_tgt @ H_gt @ np.linalg.inv(S_ref)
        seq = f"{HPATCHES_SEQ} (idx={HPATCHES_TARGET})"
        print(f"Using HPatches sequence: {seq}")
    except FileNotFoundError:
        USE_HPATCHES = False
        print("HPatches not found — run ./tools/scripts/setup_hpatches.sh first.")
        print("Falling back to synthetic warp.")

if not USE_HPATCHES:
    # Fallback: windmill with a strong synthetic warp (10% perturbation)
    wm_path = images.windmill()
    img_ref = np.array(Image.open(wm_path).convert("L"))
    img_ref, _ = _downscale(img_ref, MAX_DIM)
    img_tgt, H_gt = _synth_homography(img_ref)
    seq = "windmill (synthetic warp, 10%)"
    print(f"Using windmill with synthetic homography (10% perturbation)")

print(f"Reference: {img_ref.shape}  Target: {img_tgt.shape}")

## 3. Detect Line Segments

In [ ]:
det_ref = le_lsd.LsdCC()
det_ref.detect(img_ref)
seg_ref = det_ref.line_segments()

det_tgt = le_lsd.LsdCC()
det_tgt.detect(img_tgt)
seg_tgt = det_tgt.line_segments()

print(f"Reference segments: {len(seg_ref)}")
print(f"Target segments:    {len(seg_tgt)}")

## 4. Compute Descriptors

We compute **LBD** descriptors using the **native Sobel gradients** from
the LSD detector — these are `int16` images with typical values ±hundreds,
matching the intended input range of the C++ `FdcLBD` creator.

In [ ]:
# Use the native Sobel gradients from LSD (int16, full dynamic range).
# The LSD detector stores [gx, gy, magnitude, edge_map, segment_map]
# via image_data(). Using these instead of re-computing Sobel on float32/255
# gives proper gradient magnitudes for the LBD band statistics.
# Convert to float32 because the Python FdcLBD binding expects float type.
gx_ref = det_ref.image_data()[0].astype(np.float32)  # int16 → float32
gy_ref = det_ref.image_data()[1].astype(np.float32)
gx_tgt = det_tgt.image_data()[0].astype(np.float32)
gy_tgt = det_tgt.image_data()[1].astype(np.float32)

# Create LBD descriptors — measure wall-clock time
t0 = time.perf_counter()
lbd_ref = le_lfd.FdcLBD(gx_ref, gy_ref)
desc_ref = lbd_ref.create_list(seg_ref)
lbd_tgt = le_lfd.FdcLBD(gx_tgt, gy_tgt)
desc_tgt = lbd_tgt.create_list(seg_tgt)
t_lbd_desc = time.perf_counter() - t0

print(f"LBD descriptor dimension: {lbd_ref.size()}")
print(f"Gradient dtype: {gx_ref.dtype}, range: [{gx_ref.min()}, {gx_ref.max()}]")
print(f"Reference descriptors: {len(desc_ref)}")
print(f"Target descriptors:    {len(desc_tgt)}")
print(f"LBD descriptor time:   {t_lbd_desc*1000:.1f} ms")

## 5. Brute-Force Matching with Cross-Check & Ratio Test

We apply two standard filters to keep only high-quality matches:
- **Cross-check**: only keep mutual nearest neighbors (A→B and B→A agree)
- **Ratio test** (Lowe, 2004): reject ambiguous matches where the best
  distance is close to the second-best

In [ ]:
# Forward matching: ref → tgt (also get 2-NN for ratio test) — measure time
t0 = time.perf_counter()
matcher_fwd = le_lfd.BruteForceLBD()
matcher_fwd.train(desc_ref, desc_tgt)
fwd_best = matcher_fwd.best()
fwd_knn = matcher_fwd.knn(2)

# Backward matching: tgt → ref (for cross-check)
matcher_bwd = le_lfd.BruteForceLBD()
matcher_bwd.train(desc_tgt, desc_ref)
bwd_best = matcher_bwd.best()
t_lbd_match = time.perf_counter() - t0

# Apply cross-check + ratio test + minimum length filter
matches_raw = []  # unfiltered best matches (for comparison)
matches_cc = []   # cross-check + ratio test filtered

for i, m in enumerate(fwd_best):
    if np.isnan(m.distance):
        continue
    # Min segment length filter
    if seg_ref[i].length < MIN_SEG_LENGTH or seg_tgt[m.match_idx].length < MIN_SEG_LENGTH:
        continue
    matches_raw.append((i, m))
    # Cross-check: must be mutual nearest neighbor
    j = m.match_idx
    if j >= len(bwd_best) or bwd_best[j].match_idx != i:
        continue
    # Ratio test: best / second-best distance
    nn0 = fwd_knn[2 * i]
    nn1 = fwd_knn[2 * i + 1]
    if np.isnan(nn1.distance) or nn1.distance <= 0:
        continue
    if nn0.distance / nn1.distance >= RATIO_THRESH:
        continue
    matches_cc.append((i, m))

matches_raw.sort(key=lambda p: p[1].distance)
matches_cc.sort(key=lambda p: p[1].distance)

print(f"Total valid matches (min length {MIN_SEG_LENGTH}px): {len(matches_raw)}")
print(f"After cross-check + ratio test (thresh={RATIO_THRESH}): {len(matches_cc)}")
if matches_cc:
    dists = [m.distance for _, m in matches_cc]
    print(f"Distance range: [{dists[0]:.4f}, {dists[-1]:.4f}]")
    print(f"Median distance: {np.median(dists):.4f}")
print(f"LBD matching time: {t_lbd_match*1000:.1f} ms")

## 6. Ground-Truth Evaluation

Since we have the homography `H_gt`, we can measure which matches are
geometrically correct.  A match is **correct** if the projected reference
segment midpoint lands within 15 px of the matched target segment midpoint.

In [ ]:
GT_THRESHOLD = 15.0  # pixels — match is "correct" if projected error < this

def match_error(seg_r, seg_t, H):
    """Compute reprojection error for a match using the GT homography."""
    cx, cy = seg_r.center_point()
    pt_h = np.array([cx, cy, 1.0])
    proj = H @ pt_h
    proj = proj[:2] / proj[2]
    tx, ty = seg_t.center_point()
    return float(np.linalg.norm(proj - np.array([tx, ty])))

def classify_matches(match_list, segs_r, segs_t, H, thresh=GT_THRESHOLD):
    """Return (errors, is_correct) arrays for a match list."""
    errors = []
    correct = []
    for ref_idx, m in match_list:
        err = match_error(segs_r[ref_idx], segs_t[m.match_idx], H)
        errors.append(err)
        correct.append(err < thresh)
    return np.array(errors), np.array(correct)

# Evaluate raw matches
errs_raw, ok_raw = classify_matches(matches_raw, seg_ref, seg_tgt, H_gt)
# Evaluate cross-check + ratio test matches
errs_cc, ok_cc = classify_matches(matches_cc, seg_ref, seg_tgt, H_gt)

n_raw = min(N_EVAL, len(matches_raw))
n_cc = min(N_EVAL, len(matches_cc))
pct_raw = ok_raw[:n_raw].sum() / n_raw * 100 if n_raw else 0
pct_cc = ok_cc[:n_cc].sum() / n_cc * 100 if n_cc else 0

print(f"Raw best-match (top {n_raw}):        {ok_raw[:n_raw].sum()}/{n_raw} correct ({pct_raw:.0f}%)")
print(f"Cross-check + ratio (top {n_cc}): {ok_cc[:n_cc].sum()}/{n_cc} correct ({pct_cc:.0f}%)")
print(f"(threshold: {GT_THRESHOLD:.0f} px reprojection error)")

## 7. Visualize with Rerun

We show the **top 100 matches** (sorted by descriptor distance) so that
both correct and incorrect matches are visible, especially for harder pairs.

Matches are **color-coded by correctness**:
- **Green** — correct match (reprojection error < 15 px)
- **Red** — incorrect match

Use the **timeline scrubber** (bottom bar) to switch between views:
`view=0` detected segments, `view=1` raw matches, `view=2` cross-check + ratio matches,
`view=3` OpenCV LBD matches.

In [ ]:
import rerun.blueprint as rrb

rr.init("line_feature_demo")

COLOR_CORRECT = [0, 200, 0]    # green
COLOR_WRONG = [220, 40, 40]    # red

def generate_colors(n: int) -> np.ndarray:
    """Generate N visually distinct colors via HSV spacing."""
    colors = []
    for i in range(n):
        h = i / max(n, 1)
        r, g, b = colorsys.hsv_to_rgb(h, 0.9, 0.95)
        colors.append([int(r * 255), int(g * 255), int(b * 255)])
    return np.array(colors, dtype=np.uint8)

def gt_colors(is_correct: np.ndarray) -> np.ndarray:
    """Return green/red colors based on correctness array."""
    return np.array([COLOR_CORRECT if c else COLOR_WRONG for c in is_correct],
                    dtype=np.uint8)

def seg_to_strip(seg):
    """Convert LineSegment to [[x1,y1],[x2,y2]]."""
    x1, y1, x2, y2 = seg.end_points()
    return [[x1, y1], [x2, y2]]

def log_segments(entity: str, segments, color=None):
    """Log a list of segments as LineStrips2D."""
    strips = [seg_to_strip(s) for s in segments]
    if color is not None:
        rr.log(entity, rr.LineStrips2D(strips, colors=[color] * len(strips), radii=1.0))
    else:
        colors = generate_colors(len(strips))
        rr.log(entity, rr.LineStrips2D(strips, colors=colors, radii=1.0))

def log_matches(entity_ref, entity_tgt, match_list, segs_r, segs_t,
                colors, n_show=30):
    """Log matched segment pairs with given colors."""
    n = min(n_show, len(match_list))
    ref_strips = [seg_to_strip(segs_r[match_list[i][0]]) for i in range(n)]
    tgt_strips = [seg_to_strip(segs_t[match_list[i][1].match_idx]) for i in range(n)]
    rr.log(entity_ref, rr.LineStrips2D(ref_strips, colors=colors[:n], radii=2.5))
    rr.log(entity_tgt, rr.LineStrips2D(tgt_strips, colors=colors[:n], radii=2.5))

# Entity paths that need clearing between timeline steps.
_MATCH_ENTITIES = [
    "reference/segments", "target/segments",
    "reference/matches_raw", "target/matches_raw",
    "reference/matches_cc", "target/matches_cc",
    "reference/matches_ocv", "target/matches_ocv",
]

def clear_overlays():
    """Clear all segment/match overlays at the current timestep."""
    for e in _MATCH_ENTITIES:
        rr.log(e, rr.Clear(recursive=False))

In [ ]:
# --- Step 0: Show detected segments ---
rr.set_time("view", sequence=0)
clear_overlays()

rr.log("reference/image", rr.Image(img_ref))
log_segments("reference/segments", seg_ref)
rr.log("reference/info", rr.TextLog(f"{len(seg_ref)} segments detected"))

rr.log("target/image", rr.Image(img_tgt))
log_segments("target/segments", seg_tgt)
rr.log("target/info", rr.TextLog(f"{len(seg_tgt)} segments detected"))

In [ ]:
# --- Step 1: Show raw matches (green=correct, red=incorrect) ---
rr.set_time("view", sequence=1)
clear_overlays()

N_SHOW = min(N_EVAL, len(matches_raw))
colors_raw_gt = gt_colors(ok_raw[:N_SHOW])

rr.log("reference/image", rr.Image(img_ref))
rr.log("target/image", rr.Image(img_tgt))

log_matches("reference/matches_raw", "target/matches_raw",
            matches_raw, seg_ref, seg_tgt, colors_raw_gt, N_SHOW)

n_ok = int(ok_raw[:N_SHOW].sum())
rr.log("logs", rr.TextLog(
    f"Raw matches (top {N_SHOW}): {n_ok} correct, "
    f"{N_SHOW - n_ok} incorrect — green=correct, red=incorrect"))

In [ ]:
# --- Step 2: Show cross-check + ratio test matches (green=correct, red=incorrect) ---
rr.set_time("view", sequence=2)
clear_overlays()

N_CC = min(N_EVAL, len(matches_cc))
colors_cc_gt = gt_colors(ok_cc[:N_CC])

rr.log("reference/image", rr.Image(img_ref))
rr.log("target/image", rr.Image(img_tgt))

log_matches("reference/matches_cc", "target/matches_cc",
            matches_cc, seg_ref, seg_tgt, colors_cc_gt, N_CC)

n_ok_cc = int(ok_cc[:N_CC].sum())
rr.log("logs", rr.TextLog(
    f"Cross-check + ratio (top {N_CC}): {n_ok_cc} correct, "
    f"{N_CC - n_ok_cc} incorrect — green=correct, red=incorrect"))

### Step 3 — OpenCV LBD (BinaryDescriptor, C++ integration)

The lfd library includes `FdcOpenCVLBD` — a C++ wrapper around OpenCV's
`BinaryDescriptor` that computes **256-bit binary LBD** descriptors.  It
converts our LSD segments to OpenCV `KeyLine` format internally, computes
descriptors, and returns `FdOpenCVLBD` objects that work with our
`BruteForceOpenCVLBD` matcher (Hamming distance).

In [ ]:
# --- Step 3: OpenCV LBD (BinaryDescriptor) via C++ bindings ---
# Uses FdcOpenCVLBD from the lfd library — a C++ wrapper around OpenCV's
# BinaryDescriptor that computes 256-bit binary LBD descriptors directly
# from our LSD line segments. Matching uses Hamming distance.
import os, io
os.environ["OPENCV_LOG_LEVEL"] = "SILENT"

rr.set_time("view", sequence=3)
clear_overlays()

# Create binary LBD descriptors — measure time
_se = sys.stderr
sys.stderr = io.StringIO()
t0 = time.perf_counter()
try:
    ocv_ref = le_lfd.FdcOpenCVLBD(img_ref)
    ocv_desc_ref = ocv_ref.create_list(seg_ref)
    ocv_tgt = le_lfd.FdcOpenCVLBD(img_tgt)
    ocv_desc_tgt = ocv_tgt.create_list(seg_tgt)
finally:
    sys.stderr = _se
t_ocv_desc = time.perf_counter() - t0

# Forward + backward matching — measure time
t0 = time.perf_counter()
matcher_ocv_fwd = le_lfd.BruteForceOpenCVLBD()
matcher_ocv_fwd.train(ocv_desc_ref, ocv_desc_tgt)
ocv_fwd_best = matcher_ocv_fwd.best()

matcher_ocv_bwd = le_lfd.BruteForceOpenCVLBD()
matcher_ocv_bwd.train(ocv_desc_tgt, ocv_desc_ref)
ocv_bwd_best = matcher_ocv_bwd.best()
t_ocv_match = time.perf_counter() - t0

# Cross-check filter
matches_ocv = []
for i, m in enumerate(ocv_fwd_best):
    if np.isnan(m.distance) or m.distance > 1e10:
        continue
    if seg_ref[i].length < MIN_SEG_LENGTH or seg_tgt[m.match_idx].length < MIN_SEG_LENGTH:
        continue
    j = m.match_idx
    if j < len(ocv_bwd_best) and ocv_bwd_best[j].match_idx == i:
        matches_ocv.append((i, m))
matches_ocv.sort(key=lambda p: p[1].distance)

errs_ocv, ok_ocv = classify_matches(matches_ocv, seg_ref, seg_tgt, H_gt)
N_OCV = min(N_EVAL, len(matches_ocv))
colors_ocv_gt = gt_colors(ok_ocv[:N_OCV])

rr.log("reference/image", rr.Image(img_ref))
rr.log("target/image", rr.Image(img_tgt))

log_matches("reference/matches_ocv", "target/matches_ocv",
            matches_ocv, seg_ref, seg_tgt, colors_ocv_gt, N_OCV)

n_ok_ocv = int(ok_ocv[:N_OCV].sum())
rr.log("logs", rr.TextLog(
    f"OpenCV LBD cross-check (top {N_OCV}): {n_ok_ocv} correct, "
    f"{N_OCV - n_ok_ocv} incorrect — green=correct, red=incorrect"))

non_empty_ref = sum(1 for d in ocv_desc_ref if d.data.size > 0)
non_empty_tgt = sum(1 for d in ocv_desc_tgt if d.data.size > 0)
print(f"OpenCV LBD descriptor: 256-bit binary ({ocv_ref.size()} bytes, Hamming distance)")
print(f"Valid descriptors: ref={non_empty_ref}/{len(seg_ref)}, tgt={non_empty_tgt}/{len(seg_tgt)}")
print(f"Cross-check matches (min {MIN_SEG_LENGTH}px): {len(matches_ocv)}")
print(f"OpenCV LBD top {N_OCV} accuracy: {n_ok_ocv}/{N_OCV}")
print(f"OpenCV LBD descriptor time: {t_ocv_desc*1000:.1f} ms")
print(f"OpenCV LBD matching time:   {t_ocv_match*1000:.1f} ms")

In [ ]:
# Render the Rerun viewer inline with an explicit layout so line overlays
# are drawn on top of the images (Rerun auto-layout may put them in
# separate views).
blueprint = rrb.Blueprint(
    rrb.Horizontal(
        rrb.Spatial2DView(name="Reference", origin="reference"),
        rrb.Spatial2DView(name="Target", origin="target"),
    ),
    collapse_panels=True,
)
rr.notebook_show(width=1600, height=900, blueprint=blueprint)

## 8. Summary & Statistics

In [ ]:
print(f"{'='*65}")
print(f"Line Feature Matching Summary")
print(f"{'='*65}")
print(f"Image pair:    {seq}")
print(f"Reference:     {img_ref.shape[1]}x{img_ref.shape[0]}  ({len(seg_ref)} segments)")
print(f"Target:        {img_tgt.shape[1]}x{img_tgt.shape[0]}  ({len(seg_tgt)} segments)")
print(f"Min seg length: {MIN_SEG_LENGTH} px")
print(f"Ratio thresh:   {RATIO_THRESH}")
print(f"GT threshold:   {GT_THRESHOLD:.0f} px")
print()
print(f"Match Quality (top {N_EVAL}, <{GT_THRESHOLD:.0f}px error = correct):")
print(f"  Raw best-match:      {int(ok_raw[:n_raw].sum()):>3d}/{n_raw}")
print(f"  Cross-check + ratio: {int(ok_cc[:n_cc].sum()):>3d}/{n_cc}")
print(f"  OpenCV LBD (binary): {int(ok_ocv[:N_OCV].sum()):>3d}/{N_OCV}")
print()
print(f"Runtime Comparison:")
print(f"  {'':30s} {'Our LBD':>10s}  {'OpenCV LBD':>10s}  {'Speedup':>8s}")
print(f"  {'-'*62}")
speedup_desc = t_ocv_desc / t_lbd_desc if t_lbd_desc > 0 else float("inf")
speedup_match = t_ocv_match / t_lbd_match if t_lbd_match > 0 else float("inf")
speedup_total = (t_ocv_desc + t_ocv_match) / (t_lbd_desc + t_lbd_match) if (t_lbd_desc + t_lbd_match) > 0 else float("inf")
print(f"  {'Descriptor computation':<30s} {t_lbd_desc*1000:>8.1f}ms  {t_ocv_desc*1000:>8.1f}ms  {speedup_desc:>7.1f}x")
print(f"  {'Matching (fwd+bwd+knn)':<30s} {t_lbd_match*1000:>8.1f}ms  {t_ocv_match*1000:>8.1f}ms  {speedup_match:>7.1f}x")
t_total_lbd = t_lbd_desc + t_lbd_match
t_total_ocv = t_ocv_desc + t_ocv_match
print(f"  {'Total':<30s} {t_total_lbd*1000:>8.1f}ms  {t_total_ocv*1000:>8.1f}ms  {speedup_total:>7.1f}x")
print()
print(f"Descriptors:")
print(f"  Our LBD:    {lbd_ref.size()}-dim float  (L2 distance)")
print(f"  OpenCV LBD: 256-bit binary ({ocv_ref.size()} bytes, Hamming distance)")
print()
print(f"Total matches (all, after min-length filter):")
print(f"  Raw:        {len(matches_raw)}")
print(f"  CC+ratio:   {len(matches_cc)}")
print(f"  OpenCV CC:  {len(matches_ocv)}")
print()
print(f"Use the Rerun timeline to compare:")
print(f"  Step 0: Detected segments")
print(f"  Step 1: Raw LBD matches (green=correct, red=incorrect)")
print(f"  Step 2: Cross-check + ratio test matches")
print(f"  Step 3: OpenCV LBD (binary) matches")

## Next Steps

- **Multi-sequence benchmark** — evaluates matching accuracy across the
  curated architectural HPatches subset.  Run it as a standalone script:
  ```bash
  bazel run //evaluation/python:lbd_benchmark
  bazel run //evaluation/python:lbd_benchmark -- --idx-range 2 4 --max-dim 600
  ```
- **Parameter sweep** — compares different `numBand` / `widthBand` settings
  on the same data.  Run it as a standalone script:
  ```bash
  bazel run //evaluation/python:lbd_param_sweep
  bazel run //evaluation/python:lbd_param_sweep -- --params "11,7" "9,5" "7,5"
  ```
- **LIMAP integration** — 3D line reconstruction from multi-view stereo.

See [evaluation/python/README.md](../../evaluation/python/README.md) for
full CLI documentation.  Source code:
[evaluation/python/tools/lbd_benchmark.py](../../evaluation/python/tools/lbd_benchmark.py),
[evaluation/python/tools/lbd_param_sweep.py](../../evaluation/python/tools/lbd_param_sweep.py).